# 🏥 MediQwen — Notebook 01: Knowledge Base Setup & RAG Pipeline Validation

This notebook covers:
- **Environment & dependency check**
- **Knowledge base ingestion** via `MedicalKnowledgeBase`
- **Vector store population** via `MedicalVectorStore`
- **RAG pipeline validation** — similarity search, hybrid search, retriever


## 🔧 Path Setup & Project Root

In [1]:
import sys
import os
from pathlib import Path

# ── Adjust to project root ──────────────────────────────────
PROJECT_ROOT = Path("../").resolve()   # notebooks/ is one level below root

print(f"📁 Project root  : {PROJECT_ROOT}")
print(f"📁 Working dir   : {Path.cwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")

# Validate key source files exist
required_files = [
    PROJECT_ROOT / "src" / "rag" / "knowledge_base.py",
    PROJECT_ROOT / "src" / "rag" / "vector_store.py",
    PROJECT_ROOT / "config" / "settings.py",
    PROJECT_ROOT / "config" / "skills.md",
    PROJECT_ROOT / "src" / "agent" / "graph.py",
    PROJECT_ROOT / "src" / "agent" / "nodes.py",
    PROJECT_ROOT / "src" / "agent" / "state.py",
    PROJECT_ROOT / "src" / "tools" / "web_scraper.py",
    PROJECT_ROOT / "src" / "tools" / "system_prompt.py",
    PROJECT_ROOT / "src" / "preprocessing" / "image_processor.py",
    PROJECT_ROOT / "src" / "models" / "embeddings.py",
    PROJECT_ROOT / "src" / "safety" / "guardrails.py",
    PROJECT_ROOT / "src" / "safety" / "risk_classifier.py",
    PROJECT_ROOT / "app" / "api_server.py",
    PROJECT_ROOT / "app" / "streamlit_interface.py",
]

for f in required_files:
    status = "✅" if f.exists() else "❌ MISSING"
    print(f"  {status}  {f.relative_to(PROJECT_ROOT)}")

📁 Project root  : C:\santosh\MediQwen
📁 Working dir   : c:\santosh\MediQwen\notebooks
🐍 Python version: 3.14.2
  ✅  src\rag\knowledge_base.py
  ✅  src\rag\vector_store.py
  ✅  config\settings.py
  ✅  config\skills.md
  ✅  src\agent\graph.py
  ✅  src\agent\nodes.py
  ✅  src\agent\state.py
  ✅  src\tools\web_scraper.py
  ✅  src\tools\system_prompt.py
  ✅  src\preprocessing\image_processor.py
  ✅  src\models\embeddings.py
  ✅  src\safety\guardrails.py
  ✅  src\safety\risk_classifier.py
  ✅  app\api_server.py
  ✅  app\streamlit_interface.py


## 🔍 — Library Version Check

In [ ]:
import importlib

packages = {
    "chromadb": "chromadb",
    "langchain": "langchain",
    "langchain_community": "langchain_community",
    "langchain_huggingface": "langchain_huggingface",
    "sentence_transformers": "sentence_transformers",
    "torch": "torch",
    
}

print("📦 Installed package versions:")
print("-" * 40)
all_ok = True
for display_name, import_name in packages.items():
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"  ✅ {display_name:<30} {version}")
    except ImportError:
        print(f"  ❌ {display_name:<30} NOT INSTALLED")
        all_ok = False

print("-" * 40)
print("✅ All packages ready" if all_ok else "❌ Some packages missing — re-run Cell 1")

📦 Installed package versions:
----------------------------------------
  ✅ chromadb                       1.5.9
  ✅ langchain                      1.3.1
  ✅ langchain_community            0.4.2


c:\santosh\MediGemma\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✅ langchain_huggingface          unknown
  ✅ sentence_transformers          5.5.1
  ✅ torch                          2.12.0+cpu
----------------------------------------
✅ All packages ready


## 📚 Ingest Documents into Knowledge Base

In [3]:
import os
from pathlib import Path
from rag.knowledge_base import MedicalKnowledgeBaseBuilder
from rag.vector_store import MedicalVectorStore

# Define your project directory tracking path
KB_ROOT = Path(PROJECT_ROOT) / "data" / "knowledge_base"

print("🏥 Starting MediQwen Nested Directory Ingestion Loop...")

if not KB_ROOT.exists():
    print(f"❌ Error: Data directory target '{KB_ROOT}' does not exist on disk.")
else:
    # 1. Discover all markdown files across all subfolders recursively
    target_files = list(KB_ROOT.rglob("*.md"))
    
    if target_files:
        print(f"🔍 Discovered {len(target_files)} condition markdown file(s) scheduled for parsing.")
        
        # 2. Initialize clean builder
        kb_builder = MedicalKnowledgeBaseBuilder()
        
        # 3. Loop through every discovered file one by one
        for file_path in target_files:
            print(f"  📄 Processing: {file_path.relative_to(KB_ROOT)}")
            kb_builder.parse_markdown(file_path)
            
        # Print summary statistics across all ingested markdown data chunks
        kb_builder.print_statistics()
        
        # 4. Initialize Vector Store and stream everything directly into ChromaDB in one batch
        print("\n📥 Streaming all parsed chunks into local ChromaDB collection...")
        vector_store = MedicalVectorStore()
        
        # Retrieve all formatted chunks from the knowledge base builder and add them to the vector store
        all_chunks = kb_builder.get_all_formatted_chunks()
        success = vector_store.add_documents(all_chunks)
        
        if success:
            print("✅ Database synchronization complete!")
            # Print updated stats straight from ChromaDB's persistence layer to verify
            db_stats = vector_store.get_collection_stats()
            print(f"📊 Current database total chunk count: {db_stats.get('count', 0)}")
        else:
            print("❌ Failed to commit some chunks to the Vector Store.")
            
    else:  
        print(f"⚠️ No markdown files (.md) found inside '{KB_ROOT}' or its subfolders.")

c:\santosh\MediQwen\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-06 20:26:16,391 - INFO - Initializing Persistent ChromaDB client at: C:\santosh\MediQwen\chroma_db


🏥 Starting MediQwen Nested Directory Ingestion Loop...
🔍 Discovered 82 condition markdown file(s) scheduled for parsing.
  📄 Processing: cardiovascular\angina.md
  📄 Processing: cardiovascular\cardiomyopathy.md
  📄 Processing: cardiovascular\chest_pain.md
  📄 Processing: cardiovascular\coronary_heart_diesease.md
  📄 Processing: cardiovascular\heart_attack.md
  📄 Processing: cardiovascular\high_blood_pressure.md
  📄 Processing: cardiovascular\low_blood_pressure.md
  📄 Processing: cardiovascular\mini_stroke.md
  📄 Processing: cardiovascular\stoke.md
  📄 Processing: digestive\acid_reflux.md
  📄 Processing: digestive\constipation.md
  📄 Processing: digestive\crohn's_disease.md
  📄 Processing: digestive\gallstones.md
  📄 Processing: digestive\gastritis.md
  📄 Processing: digestive\indigestion.md
  📄 Processing: digestive\irritable_bowel_syndrome.md
  📄 Processing: digestive\ulcerative_colitis.md
  📄 Processing: first_aid\first_aid.md
  📄 Processing: health\family_planning.md
  📄 Processing:

2026-08-06 20:26:56,374 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-06 20:26:56,394 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-08-06 20:26:56,617 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-08-06 20:26:56,644 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-08-06 20:26:56,667 - INFO - Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
2026-08-06 20:26:56,911 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

✅ Database synchronization complete!
📊 Current database total chunk count: 628


## 🔍 RAG Pipeline Validation: Similarity Search

In [2]:
from rag.knowledge_base import MedicalKnowledgeBaseBuilder
from rag.vector_store import MedicalVectorStore
vector_store = MedicalVectorStore()

c:\santosh\MediQwen\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-18 20:54:59,661 - INFO - Initializing Persistent ChromaDB client at: C:\santosh\MediQwen\chroma_db


🔍 Initializing embedding model: BAAI/bge-large-en-v1.5
🔧 Using device: auto


2026-08-18 20:55:02,705 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-18 20:55:02,725 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-08-18 20:55:02,943 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-08-18 20:55:02,983 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-08-18 20:55:02,987 - INFO - Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
2026-08-18 20:55:03,624 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

In [3]:
# Test queries covering each knowledge category
TEST_QUERIES = [

    # Symptoms
    ("high fever and chills", None),
    ("severe headache", None),
    ("persistent cough", None),
    ("abdominal pain after eating", None),

    # Respiratory
    ("asthma attack treatment", None),
    ("pneumonia symptoms", None),

    # Cardiovascular
    ("hypertension treatment", None),
    ("heart attack", None),

    # First Aid
    ("first aid burn treatment steps", None),
    ("nosebleed", None),

    # Pediatric
    ("child fever", None),

    # Infectious Disease
    ("malaria symptoms", None),
    ("tuberculosis", None),
]

print("=" * 65)
print("🔍 SIMILARITY SEARCH VALIDATION")
print("=" * 65)

all_passed = True
for query, where_filter in TEST_QUERIES:
    filter_label = f" [filter: {where_filter}]" if where_filter else ""
    print(f"\n📝 Query: '{query}'{filter_label}")
    print("-" * 55)
    
    results = vector_store.similarity_search(query, k=3, score_threshold=0.75)
    
    if not results:
        print("  ❌ No results returned")
        all_passed = False
        continue
    
    for i, doc in enumerate(results):
        print(f"  [{i+1}] Source : {doc.metadata.get('source', 'unknown')}")
        print(f"       Domain : {doc.metadata.get('domain', 'unknown')}")
        print(f"       Trust  : {doc.metadata.get('trust_level', '?')}")
        print(f"       Title  : {doc.metadata.get('doc_title')}")
        print(f"       Text   : {doc.page_content[:100].strip()}...")
        print()

print("=" * 65)
print("✅ Similarity search validation PASSED" if all_passed else "⚠️  Some queries returned no results")

🔍 SIMILARITY SEARCH VALIDATION

📝 Query: 'high fever and chills'
-------------------------------------------------------


2026-08-06 21:07:13,625 - INFO - 🎯 Chunk match candidate distance score: 0.5806 for fever
2026-08-06 21:07:13,626 - INFO - 🎯 Chunk match candidate distance score: 0.6065 for cellulitis
2026-08-06 21:07:13,627 - INFO - 🎯 Chunk match candidate distance score: 0.6529 for back pain
2026-08-06 21:07:13,627 - INFO - 🔍 Validated 3/3 chunks above confidence requirements.
2026-08-06 21:07:13,703 - INFO - 🎯 Chunk match candidate distance score: 0.5302 for headaches
2026-08-06 21:07:13,704 - INFO - 🎯 Chunk match candidate distance score: 0.5688 for headaches
2026-08-06 21:07:13,705 - INFO - 🎯 Chunk match candidate distance score: 0.5861 for headaches
2026-08-06 21:07:13,705 - INFO - 🔍 Validated 3/3 chunks above confidence requirements.
2026-08-06 21:07:13,771 - INFO - 🎯 Chunk match candidate distance score: 0.5149 for cough
2026-08-06 21:07:13,772 - INFO - 🎯 Chunk match candidate distance score: 0.5499 for lung cancer
2026-08-06 21:07:13,773 - INFO - 🎯 Chunk match candidate distance score: 0.5718

  [1] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Adults

### What is a High Temperature?
- Normal body temperature varies by person and time of da...

  [2] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## When to Seek Help
- **Urgent GP/111:** painful, hot, swollen skin.
- **999 / A&E:**  
  - Very hi...

  [3] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Urgent GP / NHS 111
Seek urgent help if:
- Feeling hot, cold, shivery, or generally unwell
- Seve...


📝 Query: 'severe headache'
-------------------------------------------------------
  [1] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Causes
Common causes:
- Cold or flu
- Stress
- Alcohol
- Poor posture
- Eyesight problems
- Skipp...

  [2] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Eme

2026-08-06 21:07:13,845 - INFO - 🎯 Chunk match candidate distance score: 0.5647 for indigestion (dyspepsia)
2026-08-06 21:07:13,846 - INFO - 🎯 Chunk match candidate distance score: 0.5806 for irritable bowel syndrome (IBS)
2026-08-06 21:07:13,847 - INFO - 🎯 Chunk match candidate distance score: 0.5871 for indigestion (dyspepsia)
2026-08-06 21:07:13,847 - INFO - 🔍 Validated 3/3 chunks above confidence requirements.
2026-08-06 21:07:13,924 - INFO - 🎯 Chunk match candidate distance score: 0.5117 for asthma
2026-08-06 21:07:13,925 - INFO - 🎯 Chunk match candidate distance score: 0.5432 for asthma
2026-08-06 21:07:13,927 - INFO - 🎯 Chunk match candidate distance score: 0.5644 for asthma
2026-08-06 21:07:13,927 - INFO - 🔍 Validated 3/3 chunks above confidence requirements.
2026-08-06 21:07:13,995 - INFO - 🎯 Chunk match candidate distance score: 0.5710 for pneumonia
2026-08-06 21:07:13,996 - INFO - 🎯 Chunk match candidate distance score: 0.6051 for lung cancer
2026-08-06 21:07:13,997 - INFO -

  [1] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Symptoms
- Heartburn (burning chest pain after eating)
- Feeling full and bloated
- Nausea
- Burp...

  [2] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Main Symptoms
- Stomach pain/cramps (worse after eating, better after pooing)
- Bloating
- Diarrh...

  [3] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Overview
- Indigestion is discomfort or burning in the chest or upper abdomen after eating or dri...


📝 Query: 'asthma attack treatment'
-------------------------------------------------------
  [1] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Asthma Attack Management
- Sit upright, stay calm.
- Use reliever inhaler:
  - Blue inhaler: 1 pu...

  [2] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   

...

  [1] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Treatments
- No cure, but symptoms can be managed.
- **Lifestyle changes:** healthy diet, gentle...

  [2] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Treatments
- **Medicines:** prescribed if blood pressure is very high, if risk factors present, o...

  [3] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Notes
- Hypertension is a silent but serious condition.
- Regular checks are essential, especiall...


📝 Query: 'heart attack'
-------------------------------------------------------
  [1] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Overview
- A heart attack occurs when blood flow to the heart is blocked.
- Life-threatening emer...

  [2] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Notes
-

2026-08-06 21:07:14,350 - INFO - 🎯 Chunk match candidate distance score: 0.7979 for first aid emergencies
2026-08-06 21:07:14,351 - INFO - 🎯 Chunk match candidate distance score: 0.8078 for varicose eczema (stasis eczema)
2026-08-06 21:07:14,352 - INFO - 🔍 Validated 1/3 chunks above confidence requirements.
2026-08-06 21:07:14,433 - INFO - 🎯 Chunk match candidate distance score: 0.7145 for allergic rhinitis
2026-08-06 21:07:14,435 - INFO - 🎯 Chunk match candidate distance score: 0.7737 for diarrhoea and vomiting
2026-08-06 21:07:14,436 - INFO - 🎯 Chunk match candidate distance score: 0.8335 for toothache
2026-08-06 21:07:14,437 - INFO - 🔍 Validated 1/3 chunks above confidence requirements.
2026-08-06 21:07:14,525 - INFO - 🎯 Chunk match candidate distance score: 0.5308 for fever
2026-08-06 21:07:14,527 - INFO - 🎯 Chunk match candidate distance score: 0.5749 for fever
2026-08-06 21:07:14,528 - INFO - 🎯 Chunk match candidate distance score: 0.6272 for rashes in babies and children
2026-08

  [1] Source : NHS UK
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Common Emergencies

### Anaphylaxis
- Severe allergic reaction (food, insect sting).  
- Symptoms...


📝 Query: 'nosebleed'
-------------------------------------------------------
  [1] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## GP Treatments
- Prescription steroid nasal sprays or antihistamines
- Referral to specialist if s...


📝 Query: 'child fever'
-------------------------------------------------------
  [1] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Children

### What is a High Temperature?
- Very common in young children.
- Usually returns to n...

  [2] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Notes
- Fever is a **natural immune response** in both adults and children.  
- Most cases resolv...

  [3] Source : NHS
       D

2026-08-06 21:07:14,605 - INFO - 🎯 Chunk match candidate distance score: 0.4196 for malaria
2026-08-06 21:07:14,606 - INFO - 🎯 Chunk match candidate distance score: 0.6441 for malaria
2026-08-06 21:07:14,606 - INFO - 🎯 Chunk match candidate distance score: 0.7138 for nipah virus infection
2026-08-06 21:07:14,607 - INFO - 🔍 Validated 3/3 chunks above confidence requirements.
2026-08-06 21:07:14,742 - INFO - 🎯 Chunk match candidate distance score: 0.6471 for tuberculosis (TB)
2026-08-06 21:07:14,744 - INFO - 🎯 Chunk match candidate distance score: 0.6728 for tuberculosis (TB)
2026-08-06 21:07:14,745 - INFO - 🎯 Chunk match candidate distance score: 0.7433 for tuberculosis (TB)
2026-08-06 21:07:14,746 - INFO - 🔍 Validated 3/3 chunks above confidence requirements.


  [1] Source : WHO
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Symptoms
- Onset: 10–15 days after mosquito bite.  
- **Mild:** fever, headache, chills.  
- **Se...

  [2] Source : WHO
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Notes
- Malaria remains a leading cause of death in Africa, especially among children.  
- Vaccin...

  [3] Source : WHO
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Signs & Symptoms
- Incubation: 3–14 days (rarely up to 45).  
- Some asymptomatic; most develop f...


📝 Query: 'tuberculosis'
-------------------------------------------------------
  [1] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Overview
- TB is an infection usually affecting the lungs.
- Can be serious if untreated, but cur...

  [2] Source : NHS
       Domain : unknown
       Trust  : ?
       Title  : None
       Text   : ## Notes


## 🔀 RAG Pipeline Validation: Hybrid Search

In [5]:
print("=" * 65)
print("🔀 HYBRID SEARCH (MMR) VALIDATION: TARGETED MEDICAL TOPICS")
print("=" * 65)

HYBRID_TEST_QUERIES = [
    "early signs of lung cancer persistent cough and chest pain",
    "polio virus transmission symptoms and paralysis risk",
    "treatment options for acne vulgaris and skincare management",
    "comprehensive sexuality education and reproductive health guidance",
    "essential guidelines for a balanced healthy diet and nutritional balance"
]

all_hybrid_passed = True

for query in HYBRID_TEST_QUERIES:
    print(f"\n📝 Query: '{query}'")
    print("-" * 55)
    
    hybrid_results = vector_store.hybrid_search(
        query=query,
        k=3,
        score_threshold=0.75,  # Threshold set to 0.75 for reliable candidates
        lambda_mult=0.7        # Favor relevance while preserving candidate diversity
    )
    
    if hybrid_results:
        print(f"✅ Returned {len(hybrid_results)} diverse results (MMR)\n")
        for i, doc in enumerate(hybrid_results):
            source = doc.metadata.get('source', 'unknown')
            domain = doc.metadata.get('domain', 'unknown')
            condition = doc.metadata.get('condition', 'unknown')
            snippet = doc.page_content[:110].replace("\n", " ").strip()
            
            print(f"  [{i+1}] Condition: {condition}")
            print(f"       Domain   : {domain}")
            print(f"       Source   : {source}")
            print(f"       Text     : {snippet}...\n")
    else:
        print("❌ Hybrid search returned no results")
        all_hybrid_passed = False

print("=" * 65)
print("✅ All targeted topic searches PASSED" if all_hybrid_passed else "⚠️ Some targeted queries returned no results")

🔀 HYBRID SEARCH (MMR) VALIDATION: TARGETED MEDICAL TOPICS

📝 Query: 'early signs of lung cancer persistent cough and chest pain'
-------------------------------------------------------


2026-08-06 21:14:10,002 - INFO - 🔍 Diverse hybrid search finalized 3 valid chunks.
2026-08-06 21:14:10,165 - INFO - 🔍 Diverse hybrid search finalized 3 valid chunks.


✅ Returned 3 diverse results (MMR)

  [1] Condition: lung cancer
       Domain   : unknown
       Source   : NHS
       Text     : ## Symptoms - Persistent cough (lasting >3 weeks or worsening) - Coughing up blood - Persistent breathlessness...

  [2] Condition: cough
       Domain   : unknown
       Source   : NHS
       Text     : ## Urgent GP / NHS 111 Seek urgent help if: - Cough is very bad or worsening quickly (hacking cough, cannot st...

  [3] Condition: chronic obstructive pulmonary disease (COPD)
       Domain   : unknown
       Source   : NHS
       Text     : ## When to Seek Help - See GP if persistent symptoms, especially if >35 and smoke or used to smoke. - Early tr...


📝 Query: 'polio virus transmission symptoms and paralysis risk'
-------------------------------------------------------
✅ Returned 3 diverse results (MMR)

  [1] Condition: polio (poliomyelitis)
       Domain   : unknown
       Source   : WHO
       Text     : ## Overview - Highly infectious viral disease

2026-08-06 21:14:10,387 - INFO - 🔍 Diverse hybrid search finalized 3 valid chunks.
2026-08-06 21:14:10,561 - INFO - 🔍 Diverse hybrid search finalized 3 valid chunks.


✅ Returned 3 diverse results (MMR)

  [1] Condition: acne
       Domain   : unknown
       Source   : NHS
       Text     : ## Self Care **Do:** - Wash affected skin ≤2 times daily with mild cleanser and lukewarm water. - Use water-ba...

  [2] Condition: acne
       Domain   : unknown
       Source   : NHS
       Text     : ## When to Seek Help - **Pharmacist:** mild acne, advice on creams/gels (e.g., benzoyl peroxide). - **GP:** mo...

  [3] Condition: acne
       Domain   : unknown
       Source   : NHS
       Text     : ## Notes - Acne cannot be cured but can be controlled. - Early treatment prevents scarring. - Emotional impact...


📝 Query: 'comprehensive sexuality education and reproductive health guidance'
-------------------------------------------------------
✅ Returned 3 diverse results (MMR)

  [1] Condition: comprehensive sexuality education (CSE)
       Domain   : unknown
       Source   : WHO / UN Technical Guidance
       Text     : ## Overview - Curriculum-based, incre

2026-08-06 21:14:10,762 - INFO - 🔍 Diverse hybrid search finalized 3 valid chunks.


✅ Returned 3 diverse results (MMR)

  [1] Condition: healthy diets
       Domain   : unknown
       Source   : WHO
       Text     : ## WHO Guidance  ### Carbohydrates - 45–75% of daily energy from unrefined carbs (whole grains, pulses, fruits...

  [2] Condition: healthy diets
       Domain   : unknown
       Source   : WHO
       Text     : ## Notes - Healthy diets are foundational for lifelong health.   - Require multisectoral collaboration (health...

  [3] Condition: coronary heart disease (CHD)
       Domain   : unknown
       Source   : NHS
       Text     : ## Prevention - Stop smoking - Eat a balanced, low-fat diet - Exercise regularly - Maintain healthy weight - L...

✅ All targeted topic searches PASSED


## 🤖 RAG Pipeline Validation: LangChain Retriever

In [11]:
print("=" * 65)
print("🤖 LANGCHAIN RETRIEVER VALIDATION")
print("=" * 65)

# Initialize retriever from vector_store with relaxed threshold (0.75) and top-k setup
retriever = vector_store.get_retriever(score_threshold=0.45, k=3)

# Test queries to validate LangChain retriever interface (.invoke)
RETRIEVER_TEST_QUERIES = [
    "symptoms of high blood pressure",
    "how to manage asthma attack using reliever inhaler",
    "healthy lifestyle and nutrition"
]

all_retriever_passed = True

for query in RETRIEVER_TEST_QUERIES:
    print(f"\n📝 Retriever Query: '{query}'")
    print("-" * 55)
    
    try:
        # Standard LangChain Runnable interface (.invoke)
        retrieved_docs = retriever.invoke(query)
        
        if retrieved_docs:
            print(f"✅ Retrieved {len(retrieved_docs)} chunk(s) via retriever\n")
            for i, doc in enumerate(retrieved_docs):
                condition = doc.metadata.get('condition', 'unknown')
                domain = doc.metadata.get('domain', 'unknown')
                source = doc.metadata.get('source', 'unknown')
                snippet = doc.page_content[:120].replace("\n", " ").strip()
                
                print(f"  [{i+1}] Condition: {condition}")
                print(f"       Domain   : {domain}")
                print(f"       Source   : {source}")
                print(f"       Text     : {snippet}...\n")
        else:
            print("  ❌ No documents met the retriever confidence threshold")
            all_retriever_passed = False

    except Exception as e:
        print(f"  ❌ Retriever call failed with error: {e}")
        all_retriever_passed = False

print("=" * 65)
print("✅ LangChain Retriever validation PASSED" if all_retriever_passed else "⚠️ Some retriever calls returned no results")

2026-08-06 21:20:49,316 - INFO - ✅ Secure safety-capped retriever spawned (threshold=0.45).


🤖 LANGCHAIN RETRIEVER VALIDATION

📝 Retriever Query: 'symptoms of high blood pressure'
-------------------------------------------------------
✅ Retrieved 3 chunk(s) via retriever

  [1] Condition: high blood pressure (hypertension)
       Domain   : unknown
       Source   : NHS
       Text     : ## Overview - High blood pressure (hypertension) increases risk of heart attack, stroke, kidney disease, and vascular de...

  [2] Condition: high blood pressure (hypertension)
       Domain   : unknown
       Source   : NHS
       Text     : ## Symptoms - Usually no symptoms. - Rarely: headaches, blurred vision, chest pain. - Only way to confirm is a blood pre...

  [3] Condition: high blood pressure (hypertension)
       Domain   : unknown
       Source   : NHS
       Text     : ## Risks Untreated high blood pressure increases risk of: - Heart disease - Heart attack - Stroke - Heart failure - Kidn...


📝 Retriever Query: 'how to manage asthma attack using reliever inhaler'
-----------------

# RAG Context Inspection

In [3]:
def inspect_retrieved_context_for_llm(query: str, top_k: int = 3, threshold: float = 0.55):
    """
    Validates exact chunk context, distance scores, and metadata 
    prior to LLM generation.
    """
    print(f"🔍 INSPECTING VECTOR DB RETRIEVAL FOR: '{query}'")
    print("=" * 70)
    
    # Run hybrid search
    results = vector_store.hybrid_search(query=query, k=top_k, score_threshold=threshold)
    
    if not results:
        print("🛑 DB MISS: No candidates cleared distance threshold bounds.")
        print("   ↳ State Router will trigger Safe Refusal or Web Fallback.")
        return
        
    for idx, doc in enumerate(results, 1):
        score = doc.metadata.get("score", "N/A")
        source = doc.metadata.get("source", "Unknown")
        condition = doc.metadata.get("condition", "General")
        
        print(f"📄 Chunk [{idx}/{len(results)}]")
        print(f"   • Similarity Score : {score}")
        print(f"   • Condition Target : {condition}")
        print(f"   • Verified Source  : {source}")
        print("   • Raw Context Payload:")
        print("   " + "-"*60)
        print(f"   {doc.page_content.strip()}")
        print("   " + "-"*60 + "\n")

# Test Retrieval Inspection
inspect_retrieved_context_for_llm("What are the core symptoms of malaria?")

🔍 INSPECTING VECTOR DB RETRIEVAL FOR: 'What are the core symptoms of malaria?'


2026-08-18 20:55:25,887 - INFO - 🔍 Diverse hybrid search finalized 1 valid chunks.


📄 Chunk [1/1]
   • Similarity Score : 0.5148
   • Condition Target : malaria
   • Verified Source  : WHO
   • Raw Context Payload:
   ------------------------------------------------------------
   ## Symptoms
- Onset: 10–15 days after mosquito bite.  
- **Mild:** fever, headache, chills.  
- **Severe:** fatigue, impaired consciousness, seizures, difficulty breathing, dark/bloody urine, jaundice, abnormal bleeding.  
- Severe malaria can cause death within 24 hours if untreated.  
- Malaria in pregnancy: premature delivery, low birth weight.
   ------------------------------------------------------------

